# 25 — MVCC Versioned Store (Amazon FAR style)

Multi-Version Concurrency Control: every write creates a new version, so readers see a consistent
**snapshot** without blocking writers — the concurrency-control model behind Postgres/MySQL-InnoDB.
Parts 1-3 build the core (concurrency at Part 3); Parts 4-5 are stretch tasks (tombstone deletes + GC,
then parallel point-in-time reads). Fill stubs, run each test cell, peek at the collapsed `ref_`
solutions only after trying.

Each key keeps an ascending list of `(version, value)`; a global counter assigns versions.

---

## Part 1 — Versioned reads

`VersionedStore` with `put(key, value) -> version` (assigns a new global, monotonically increasing
version), `get(key)` (latest value, `None` if absent), and `get_as_of(key, version)` (the value as of
that version = latest write with `ver <= version`, else `None`).

**Lock down:** versions are global and increasing; `get_as_of` reads the newest version not after the
snapshot — a key written *after* the snapshot is invisible.

In [ ]:
from collections import defaultdict


class VersionedStore:
    def __init__(self):
        raise NotImplementedError

    def put(self, key, value):
        raise NotImplementedError

    def get(self, key):
        raise NotImplementedError

    def get_as_of(self, key, version):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    s = VersionedStore()
    v1 = s.put("a", 10); v2 = s.put("a", 20); v3 = s.put("b", 5)
    assert s.get("a") == 20
    assert s.get_as_of("a", v1) == 10 and s.get_as_of("a", v2) == 20
    assert s.get_as_of("b", v1) is None                 # b didn't exist at v1
    assert s.get("zzz") is None
    print("Part 1 OK")

_t1()

## Part 2 — Snapshot view

`snapshot_view(store, version) -> dict`: a point-in-time read of the **whole** store — every key's
value as of `version` (omit keys not yet written by then).

**Lock down:** a consistent snapshot reflects all writes with `ver <= version` and none after; this is
what gives a long-running reader a stable view while writers keep going.

In [ ]:
def snapshot_view(store, version):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    s = VersionedStore()
    s.put("a", 1); s.put("b", 2); snap = s.put("a", 3)  # snapshot here: a=3, b=2
    s.put("b", 99); s.put("c", 7)                       # writes after the snapshot
    assert snapshot_view(s, snap) == {"a": 3, "b": 2}   # c absent, b is the old value
    assert snapshot_view(s, 1) == {"a": 1}
    print("Part 2 OK")

_t2()

## Part 3 — Concurrent MVCC

`ConcurrentVersionedStore`: thread-safe `put`, `get`, `current_version()`, and `snapshot_view(version)`.
The version counter must be atomic so every write gets a unique, increasing version under concurrency.

**The invariant:** 8 threads each `put` 100 times leaves `current_version() == 800` (no version reused
or skipped). **Discuss:** the atomic fetch-and-increment of the version; that readers (snapshots) never
block writers in real MVCC; append-only version lists.

In [ ]:
import threading


class ConcurrentVersionedStore:
    def __init__(self):
        raise NotImplementedError

    def put(self, key, value):
        raise NotImplementedError

    def get(self, key):
        raise NotImplementedError

    def current_version(self):
        raise NotImplementedError

    def snapshot_view(self, version):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    s = ConcurrentVersionedStore()

    def writer(t):
        for i in range(100):
            s.put("k%d" % t, i)

    ts = [threading.Thread(target=writer, args=(t,)) for t in range(8)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert s.current_version() == 800                   # unique, contiguous versions
    view = s.snapshot_view(800)
    assert len(view) == 8 and all(view["k%d" % t] == 99 for t in range(8))
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Deletes & garbage collection

`MVCCStore`: `put`/`get`/`delete` (a delete writes a **tombstone** version, so `get` returns `None`)
and `gc(keep_from)` — compact each key's history, dropping versions older than the newest one with
`ver <= keep_from` (those can never be needed by a snapshot at or after `keep_from`).

**Discuss:** tombstones vs physical deletes, why GC needs the oldest live snapshot's version as a
watermark, and bounding version-history growth.

In [ ]:
DELETED = object()


class MVCCStore:
    def __init__(self):
        raise NotImplementedError

    def put(self, key, value):
        raise NotImplementedError

    def delete(self, key):
        raise NotImplementedError

    def get(self, key):
        raise NotImplementedError

    def gc(self, keep_from):
        raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    s = MVCCStore()
    s.put("a", 1); s.put("a", 2); s.delete("a")
    assert s.get("a") is None                           # tombstoned
    s.put("a", 3); assert s.get("a") == 3               # re-added after delete

    s2 = MVCCStore()
    v = s2.put("x", 1); s2.put("x", 2); s2.put("x", 3)
    s2.gc(keep_from=v)                                  # v=1 still needs the v1 version -> keep all
    assert len(s2.data["x"]) == 3
    s2.gc(keep_from=10)                                 # nothing older than v3 is needed
    assert len(s2.data["x"]) == 1 and s2.get("x") == 3
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel point-in-time reads

`parallel_as_of(histories, queries, num_procs) -> list[(key, version, value)]`: answer a big batch of
"value as of version" reads across processes with `ProcessPoolExecutor`; worker is
`mvcc_workers.answer_queries`. `histories` is `dict[key, [(ver, val)]]` (ascending); `queries` is a
list of `(key, version)`. (Results may be in any order — match on the (key, version).)

**Discuss:** reads against an immutable version history are pure & independent (embarrassingly
parallel); shipping the history to workers; this is how analytics replays a snapshot at scale.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import mvcc_workers


def parallel_as_of(histories, queries, num_procs) -> list:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    hist = {"a": [(1, 10), (3, 20)], "b": [(2, 5)]}
    queries = [("a", 1), ("a", 3), ("a", 2), ("b", 1), ("b", 2), ("c", 5)]
    res = {(k, v): val for k, v, val in parallel_as_of(hist, queries, 3)}
    assert res[("a", 1)] == 10 and res[("a", 3)] == 20 and res[("a", 2)] == 10
    assert res[("b", 1)] is None and res[("b", 2)] == 5 and res[("c", 5)] is None
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Snapshot isolation vs serializable; write-write conflict detection (first-committer-wins / SSI).
- Read timestamps & commit timestamps; long-running readers blocking GC.
- Storage: append-only version chains vs undo logs; vacuum / compaction.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
from collections import defaultdict
import threading
from concurrent.futures import ProcessPoolExecutor
import mvcc_workers


class RefVersionedStore:
    def __init__(self):
        self.data = defaultdict(list)                   # key -> [(version, value)] ascending
        self.ver = 0

    def put(self, key, value):
        self.ver += 1
        self.data[key].append((self.ver, value))
        return self.ver

    def get(self, key):
        vs = self.data.get(key)
        return vs[-1][1] if vs else None

    def get_as_of(self, key, version):
        val = None
        for ver, v in self.data.get(key, []):
            if ver <= version:
                val = v
            else:
                break
        return val


def ref_snapshot_view(store, version):
    out = {}
    for key, versions in store.data.items():
        val = None
        for ver, v in versions:
            if ver <= version:
                val = v
            else:
                break
        if val is not None:
            out[key] = val
    return out


class RefConcurrentVersionedStore:
    def __init__(self):
        self.data = defaultdict(list)
        self.ver = 0
        self.lock = threading.Lock()

    def put(self, key, value):
        with self.lock:                                 # atomic fetch-and-increment
            self.ver += 1
            self.data[key].append((self.ver, value))
            return self.ver

    def get(self, key):
        with self.lock:
            vs = self.data.get(key)
            return vs[-1][1] if vs else None

    def current_version(self):
        with self.lock:
            return self.ver

    def snapshot_view(self, version):
        with self.lock:
            out = {}
            for key, versions in self.data.items():
                val = None
                for ver, v in versions:
                    if ver <= version:
                        val = v
                    else:
                        break
                if val is not None:
                    out[key] = val
            return out


DELETED = object()


class RefMVCCStore:
    def __init__(self):
        self.data = defaultdict(list)
        self.ver = 0

    def put(self, key, value):
        self.ver += 1
        self.data[key].append((self.ver, value))
        return self.ver

    def delete(self, key):
        self.ver += 1
        self.data[key].append((self.ver, DELETED))
        return self.ver

    def get(self, key):
        vs = self.data.get(key)
        if not vs:
            return None
        val = vs[-1][1]
        return None if val is DELETED else val

    def gc(self, keep_from):
        for key, vs in self.data.items():
            idx = 0
            for i, (ver, _) in enumerate(vs):
                if ver <= keep_from:
                    idx = i
                else:
                    break
            self.data[key] = vs[idx:]                    # keep the snapshot-visible version + newer


def ref_parallel_as_of(histories, queries, num_procs):
    chunks = [queries[i::num_procs] for i in range(num_procs)]
    out = []
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        for r in ex.map(mvcc_workers.answer_queries, [(histories, c) for c in chunks]):
            out.extend(r)
    return out


_s = RefVersionedStore(); _v1 = _s.put("a", 1); _s.put("a", 2)
assert _s.get("a") == 2 and _s.get_as_of("a", _v1) == 1
assert ref_snapshot_view(_s, _v1) == {"a": 1}
_m = RefMVCCStore(); _m.put("k", 1); _m.delete("k"); assert _m.get("k") is None
_m.put("k", 2); _m.put("k", 3); _m.gc(10); assert len(_m.data["k"]) == 1 and _m.get("k") == 3
_r = {(k, v): val for k, v, val in ref_parallel_as_of({"a": [(1, 9), (5, 8)]}, [("a", 1), ("a", 6)], 2)}
assert _r == {("a", 1): 9, ("a", 6): 8}
print("reference solutions OK")